# Variables redondantes

| Module | Durée | Prérequis |
|---|---|---|
| M5 – Espaces vectoriels et bases | 75 min | M4 |

In [ ]:
# Style graphique Liouaï : couleurs du logo, axes sobres
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap

LIOUAI = {"navy": "#243743", "vert": "#28B78D", "vert_texte": "#167A5C", "gris": "#A1A9AE",
          "gris_texte": "#5C6870", "jaune": "#E5C800", "fond": "#F3F5F6"}
plt.rcParams.update({
    "axes.prop_cycle": plt.cycler(color=[LIOUAI["vert"], LIOUAI["navy"], LIOUAI["gris"],
                                         LIOUAI["jaune"], LIOUAI["vert_texte"], LIOUAI["gris_texte"]]),
    "axes.edgecolor": LIOUAI["gris"], "axes.labelcolor": LIOUAI["navy"],
    "axes.titlecolor": LIOUAI["navy"], "axes.titleweight": "bold",
    "axes.spines.top": False, "axes.spines.right": False,
    "xtick.color": LIOUAI["gris_texte"], "ytick.color": LIOUAI["gris_texte"],
    "text.color": LIOUAI["navy"], "grid.color": "#E1E6E9", "legend.frameon": False,
    "font.family": "sans-serif", "font.sans-serif": ["Roboto", "Helvetica", "Arial", "DejaVu Sans"],
    "figure.dpi": 100,
})
# Dégradé de marque : gris clair -> vert -> navy (et sa version inversée)
if "liouai" not in mpl.colormaps:
    _cmap = LinearSegmentedColormap.from_list("liouai", [LIOUAI["fond"], LIOUAI["vert"], LIOUAI["navy"]])
    mpl.colormaps.register(_cmap)
    mpl.colormaps.register(_cmap.reversed())

## ① Définir la question

Une petite librairie en ligne, qui vend aussi des jeux de société, exporte chaque soir un tableau de ses ventes du jour. Au fil du temps, les colonnes se sont accumulées : chacun a ajouté celle dont il avait besoin. La gérante veut maintenant prévoir ses ventes avec un modèle, et son outil refuse de fonctionner : « matrice singulière ».

Au module M4, on a vu qu'une matrice est singulière quand elle écrase une direction. Ici, la cause est plus terre à terre : certaines colonnes **se calculent à partir des autres**. Elles n'apportent aucune information nouvelle.

**La question de la gérante : quelles colonnes sont redondantes, et lesquelles garder ? Combien d'informations vraiment différentes le tableau contient-il ?**

**Activité** : Avant toute analyse, quelles colonnes vous semblent suspectes, rien qu'à leur nom ?

Indiquez vos réponses et justifiez-les.

## ② Estimer avant de calculer

Le tableau compte 12 colonnes (voir ci-dessous).

In [ ]:
pari_nombre_informations_independantes = 9     # entre 1 et 12 (modifiez)

## ③ Abstraire : indépendance, rang, relations

Chaque colonne du tableau est un vecteur de $\mathbb{R}^n$, où $n$ est le nombre de jours.

- Des vecteurs sont **linéairement dépendants** si l'un d'eux est une combinaison linéaire des autres, par exemple $\mathbf{w} = 2\,\mathbf{u} + 3\,\mathbf{v}$. Sinon, ils sont **indépendants**.
- Le **rang** d'une matrice est le nombre maximal de colonnes indépendantes. C'est la **dimension** de l'espace engendré par les colonnes : le nombre d'informations réellement différentes.
- Une **relation** entre colonnes s'écrit $c_1\mathbf{a}_1 + \dots + c_p\mathbf{a}_p = \mathbf{0}$, c'est-à-dire $A\mathbf{c} = \mathbf{0}$. L'ensemble de ces vecteurs $\mathbf{c}$ s'appelle le **noyau** de $A$. Sa dimension est le nombre de colonnes redondantes :

$$
\text{nombre de colonnes} = \operatorname{rang} A + \dim \ker A .
$$

![Schéma : à gauche, w = u + v reste dans le plan de u et v (rang 2) ; à droite, w sort du plan (rang 3)](../../assets/schemas/schema-dependance.svg)

**Activité** : Soit $\mathbf{u} = (1\,;\,0\,;\,2)$, $\mathbf{v} = (0\,;\,1\,;\,1)$ et $\mathbf{w} = (2\,;\,3\,;\,7)$. Cherchez $a$ et $b$ tels que $\mathbf{w} = a\,\mathbf{u} + b\,\mathbf{v}$. Quel est le rang de la matrice de colonnes $\mathbf{u}, \mathbf{v}, \mathbf{w}$ ? Écrivez un vecteur non nul de son noyau.

Indiquez vos réponses et justifiez-les.

### Une méthode pour repérer les colonnes redondantes

On parcourt les colonnes une à une. Pour chacune, on se demande : **est-elle une combinaison des colonnes déjà gardées ?** Pour le savoir, on cherche les coefficients qui l'approchent le mieux (`np.linalg.lstsq`, étudié en détail au module M6) et on regarde l'écart restant.

- Si l'écart est nul (ou minuscule), la colonne est redondante, et les coefficients donnent la relation.
- Sinon, elle apporte une direction nouvelle : on la garde.

À la fin, le nombre de colonnes gardées est le rang.

## ④ Calculer

### Les données

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

rng = np.random.default_rng(5)
n = 90                                                     # trois mois de ventes
clients = rng.poisson(60, n)
livres = rng.binomial(clients * 2, 0.6)
jeux = rng.binomial(clients, 0.35)
ca_livres = np.round(livres * rng.normal(12.5, 1.2, n), 2)  # chiffre d'affaires hors taxes, en euros
ca_jeux = np.round(jeux * rng.normal(28, 4, n), 2)
ventes = pd.DataFrame({
    "clients": clients,
    "articles_livres": livres,
    "articles_jeux": jeux,
    "articles_total": livres + jeux,
    "ca_livres": ca_livres,
    "ca_jeux": ca_jeux,
    "ca_ht": np.round(ca_livres + ca_jeux, 2),
    "tva": np.round(0.055 * ca_livres + 0.20 * ca_jeux, 2),
    "ca_ttc": np.round(1.055 * ca_livres + 1.20 * ca_jeux, 2),
    "couts_du_jour": np.round(250 + 0.35 * (ca_livres + ca_jeux), 2),
    "panier_moyen": np.round((1.055 * ca_livres + 1.20 * ca_jeux) / clients, 2),
    "temperature": np.round(rng.normal(14, 6, n), 1),
})
ventes.head()

### Le rang, d'abord avec l'outil standard

In [ ]:
X = ventes.to_numpy(dtype=float)
print("Nombre de colonnes :", X.shape[1])
print("np.linalg.matrix_rank :", np.linalg.matrix_rank(X))

**À moitié convaincant.** L'outil trouve un rang de 10 : il repère deux colonnes redondantes sur 12. Ce sont celles dont la relation est **exacte**, car calculée sur des entiers ou sur des sommes de centimes (`articles_total`, `ca_ht`).

Mais la TVA, elle, est **arrondie au centime** : la relation `tva = 0,055 × ca_livres + 0,2 × ca_jeux` n'est vraie qu'à un demi-centime près. Or `matrix_rank` utilise une tolérance prévue pour les erreurs de calcul de l'ordinateur, de l'ordre de $10^{-16}$ en relatif, bien plus petite qu'un centime. Il nous faut une tolérance adaptée à la précision des données.

### La méthode gloutonne, avec une tolérance choisie

In [ ]:
def colonnes_independantes(tableau, tol=1e-4):
    # Renvoie la liste des colonnes gardées et, pour chaque colonne écartée, ses coefficients
    gardees, relations = [], {}
    for nom in tableau.columns:
        col = tableau[nom].to_numpy(dtype=float)
        if gardees:
            B = tableau[gardees].to_numpy(dtype=float)
            coef, *_ = np.linalg.lstsq(B, col, rcond=None)
            ecart = np.linalg.norm(col - B @ coef) / np.linalg.norm(col)   # écart relatif
        else:
            ecart = 1.0
        if not gardees or ecart > tol:
            gardees.append(nom)
        else:
            poids = np.abs(coef) * np.linalg.norm(B, axis=0) / np.linalg.norm(col)   # part de chaque terme
            relations[nom] = {g: c for g, c, p in zip(gardees, coef, poids) if p > 1e-3}   # on ignore le bruit
    return gardees, relations

def afficher(relations):
    for nom, coef in relations.items():
        termes = " ".join(f"{'−' if c < 0 else '+'} {abs(c):.4g} × {g}" for g, c in coef.items())
        print(f"  {nom} = {termes.removeprefix('+ ').replace('− ', '−', 1) if termes.startswith('−') else termes.removeprefix('+ ')}")

gardees, relations = colonnes_independantes(ventes)
print(f"Rang (à la tolérance 1e-4) : {len(gardees)}")
print("Colonnes gardées :", gardees)
print("Relations trouvées :"); afficher(relations)

### Quelle tolérance choisir ?

On trace le rang trouvé en fonction de la tolérance. Un bon choix se situe sur un **palier** : le résultat ne dépend alors presque pas de la valeur exacte.

In [ ]:
tolerances = np.logspace(-14, -1, 53)
rangs = [len(colonnes_independantes(ventes, t)[0]) for t in tolerances]

fig, ax = plt.subplots(figsize=(8, 3.5))
ax.semilogx(tolerances, rangs, marker="o", ms=3)
ax.axvspan(1e-5, 1e-2, color=LIOUAI["vert"], alpha=0.15, label="palier : arrondis au centime")
ax.axvline(1e-4, color=LIOUAI["navy"], ls="--", lw=1)
ax.set_yticks(range(6, 11)); ax.set_xlabel("tolérance sur l'écart relatif"); ax.set_ylabel("rang trouvé"); ax.legend()
ax.set_title("Le rang « numérique » dépend de la tolérance")
plt.tight_layout(); plt.show()

**Activité** : Entre $10^{-7}$ et $10^{-5}$, le rang vaut 9 : une seule des deux relations arrondies est détectée. Laquelle, à votre avis ? Pourquoi la TVA, qui porte sur de petits montants, a-t-elle une erreur **relative** d'arrondi plus grande ? Et que se passe-t-il pour une tolérance trop grande, au-delà de $10^{-2}$ ?

Indiquez vos réponses et justifiez-les.

### Deux limites de la méthode

**Une relation affine.** Les coûts du jour valent $250 + 0{,}35 \times$ le chiffre d'affaires hors taxes. À cause du terme constant 250, ce n'est pas une combinaison **linéaire** des colonnes. Pour la détecter, on ajoute une colonne constante, égale à 1 tous les jours.

**Une relation non linéaire.** Le panier moyen est `ca_ttc / clients` : c'est un quotient, pas une combinaison linéaire. Aucune méthode d'algèbre linéaire ne le révèle directement.

In [ ]:
avec_constante = ventes.copy()
avec_constante.insert(0, "constante", 1.0)
gardees_c, relations_c = colonnes_independantes(avec_constante)
print(f"Avec une colonne constante : rang {len(gardees_c)}")
print("Colonnes gardées :", gardees_c)
print("Relations :"); afficher(relations_c)

## ⑤ Interpréter

In [ ]:
print(f"Votre pari : {pari_nombre_informations_independantes} informations indépendantes")
print(f"Résultat   : {len(gardees_c)} colonnes indépendantes, dont la constante ; {len(relations_c)} colonnes redondantes")

**Activité** :

1. Lisez la relation trouvée pour `tva`. Que représentent les coefficients 0,055 et 0,2 ? La méthode a-t-elle « découvert » une règle que personne ne lui avait donnée ?
2. Rédigez pour la gérante la liste des colonnes à garder pour son modèle, en une phrase par colonne écartée.
3. Le panier moyen est gardé alors qu'il se calcule à partir d'autres colonnes. Faut-il le garder dans le modèle ? Qu'en pensez-vous ?
4. Pourquoi l'outil de prévision refusait-il de fonctionner ? Faites le lien avec le module M4.

Indiquez vos réponses et justifiez-les.

## ⑥ Critiquer et généraliser

- **Le choix n'est pas unique.** La méthode garde la **première** colonne rencontrée de chaque groupe redondant. Si l'on range les colonnes dans un autre ordre, garde-t-on les mêmes ? Le rang change-t-il ? Ce constat illustre un théorème : toutes les bases d'un même espace ont le même nombre de vecteurs, la **dimension**, mais il existe beaucoup de bases.
- **Faux positifs.** Avec seulement 5 jours de données, que se passe-t-il ? Peut-on avoir plus de 5 colonnes indépendantes dans $\mathbb{R}^5$ ?
- **Généraliser.** Écrivez une fonction `rang_selon_ordre(tableau, ordre)` qui applique la méthode après avoir rangé les colonnes dans l'ordre donné. Testez l'ordre inverse, puis 5 jours seulement.

In [ ]:
def rang_selon_ordre(tableau, ordre):
    # À compléter
    ...

## Corrigé

*À n'ouvrir qu'après avoir cherché.*

In [ ]:
def rang_selon_ordre(tableau, ordre):
    gardees, relations = colonnes_independantes(tableau[ordre])
    return len(gardees), gardees, relations

ordre_inverse = list(avec_constante.columns[::-1])
r, g, rel = rang_selon_ordre(avec_constante, ordre_inverse)
print(f"Ordre inverse : rang {r}")
print("Colonnes gardées :", g)
afficher(rel)

r5, g5, rel5 = rang_selon_ordre(avec_constante.head(5), list(avec_constante.columns))
print(f"\nAvec 5 jours seulement : rang {r5}, et {len(rel5)} « relations », presque toutes fausses :")
afficher(dict(list(rel5.items())[:3]))

## Bilan — auto-évaluation

Pour chaque compétence travaillée dans ce notebook, choisissez un niveau : **0** non observé · **1** guidé · **2** autonome · **3** transfert.
La cellule enregistre votre bilan dans `progression.json`, à la racine du site.

In [ ]:
import json, datetime, pathlib

bilan = {
    "Reconnaître si le concept s'applique": 1,
    "Relier les traits du résultat à leur sens réel": 1,
    "Identifier les concepts pertinents et leurs relations": 1,
    "Corroborer ses résultats": 1,
    "Estimer la solution avant de calculer": 1,
}   # modifiez les niveaux

ici = pathlib.Path.cwd()
racine = next((p for p in [ici, *ici.parents] if (p / "myst.yml").exists()), ici)
fichier = racine / "progression.json"
historique = json.loads(fichier.read_text(encoding="utf-8")) if fichier.exists() else []
historique.append({"notebook": "5.2-variables-redondantes", "date": datetime.date.today().isoformat(), "niveaux": bilan})
fichier.write_text(json.dumps(historique, ensure_ascii=False, indent=2), encoding="utf-8")
print(f"Bilan enregistré ({len(historique)} entrée(s) dans {fichier.name}).")